# Aneurysm Volume Prediction Baseline (3D Segmentation + CV Ensemble)

该Notebook实现：
1. 使用3D分割模型（而不是直接回归）预测动脉瘤mask。
2. 使用z-score归一化。
3. 使用数据增强（翻转、旋转、强度扰动）。
4. 使用Dice+BCE分割损失。
5. 使用K折交叉验证集成。
6. 使用NIfTI spacing计算体积(mm^3)。


In [1]:
# 如果有缺包，请取消注释安装
%pip install nibabel scikit-learn tqdm pandas matplotlib
%pip install torch torchvision torchaudio


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import random
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

# ===== 跨平台路径配置（本地 / Kaggle / 比赛平台）=====
# 可选：手动指定数据根目录（含 train/test/train_labels）
# os.environ['ANEURYSM_DATA_ROOT'] = '/Users/songling/Desktop/Aneurysm Volume Prediction'
DATA_ROOT = os.environ.get('ANEURYSM_DATA_ROOT', '').strip()

def is_valid_dataset_dir(p: Path) -> bool:
    return (p / 'train').exists() and (p / 'test').exists() and (p / 'train_labels').exists()

def find_base_dir():
    if DATA_ROOT:
        p = Path(DATA_ROOT)
        if is_valid_dataset_dir(p):
            return p

    # 常见固定路径
    fixed = [
        '/dataset/public',
        '/dataset',
        '/kaggle/input/aneurysm-volume-prediction',
        '/kaggle/input/aneurysm-volume',
        '/kaggle/working/Aneurysm Volume Prediction',
        '/Users/songling/Desktop/Aneurysm Volume Prediction',
    ]
    for c in fixed:
        p = Path(c)
        if is_valid_dataset_dir(p):
            return p

    # 自动扫描常见输入根目录
    scan_roots = [Path('/dataset'), Path('/kaggle/input'), Path.cwd()]
    for root in scan_roots:
        if not root.exists():
            continue
        for d in root.rglob('*'):
            if d.is_dir() and is_valid_dataset_dir(d):
                return d

    return None

BASE_DIR = find_base_dir()
if BASE_DIR is None:
    print('DEBUG cwd:', Path.cwd())
    for cand in ['/dataset', '/dataset/public', '/kaggle/input', '/Users/songling/Desktop']:
        cp = Path(cand)
        if cp.exists():
            print(f'DEBUG exists: {cp}')
            for x in list(cp.iterdir())[:20]:
                print(' -', x)
    raise FileNotFoundError('未找到数据目录。请设置 ANEURYSM_DATA_ROOT 到含 train/test/train_labels 的目录。')

TRAIN_IMG_DIR = BASE_DIR / 'train'
TRAIN_MASK_DIR = BASE_DIR / 'train_labels'
TEST_IMG_DIR = BASE_DIR / 'test'
TRAIN_CSV = BASE_DIR / 'train.csv'
SAMPLE_SUB_CSV = BASE_DIR / 'sample_submission.csv'

# ===== 输出目录：每次实验自动新建子目录 =====
# 评测器常见固定读取路径：/root/setup/solution/working/submission.csv
if Path('/root/setup/solution/working').exists():
    OUTPUT_ROOT = Path('/root/setup/solution/working')
elif Path('/working').exists():
    OUTPUT_ROOT = Path('/working')
elif Path('/kaggle/working').exists():
    OUTPUT_ROOT = Path('/kaggle/working')
else:
    OUTPUT_ROOT = BASE_DIR / 'working'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

RUN_NAME = os.environ.get('RUN_NAME', '').strip() or datetime.now().strftime('segcv_%Y%m%d_%H%M%S')
EXP_DIR = OUTPUT_ROOT / RUN_NAME
CKPT_DIR = EXP_DIR / 'checkpoints'
PRED_DIR = EXP_DIR / 'predictions'
LOG_DIR = EXP_DIR / 'logs'
for d in [EXP_DIR, CKPT_DIR, PRED_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Python:', sys.executable)
print('DEVICE =', DEVICE)
print('BASE_DIR =', BASE_DIR)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
print('RUN_NAME =', RUN_NAME)
print('EXP_DIR =', EXP_DIR)


Python: /Users/songling/Documents/conda/miniconda3/envs/aneurysm-seg/bin/python
DEVICE = mps
BASE_DIR = /Users/songling/Desktop/Aneurysm Volume Prediction
OUTPUT_ROOT = /Users/songling/Desktop/Aneurysm Volume Prediction/working
RUN_NAME = segcv_20260219_182619
EXP_DIR = /Users/songling/Desktop/Aneurysm Volume Prediction/working/segcv_20260219_182619


In [3]:
def load_nii(path):
    nii = nib.load(str(path))
    arr = nii.get_fdata(dtype=np.float32)
    spacing = nii.header.get_zooms()[:3]
    return arr, spacing

def zscore_norm(x, eps=1e-6):
    mean = x.mean()
    std = x.std()
    return (x - mean) / (std + eps)

def volume_from_mask(mask, spacing):
    voxel_volume = float(spacing[0] * spacing[1] * spacing[2])
    voxel_count = float(mask.sum())
    vol_mm3 = voxel_count * voxel_volume
    return max(vol_mm3, 0.0)

def volumetric_similarity(y_true, y_pred, eps=1e-4):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    vs = 1.0 - np.abs(y_true - y_pred) / (y_true + y_pred + eps)
    return float(vs.mean())


In [4]:
class AneurysmSegDataset(Dataset):
    def __init__(self, img_paths, mask_paths=None, augment=False):
        self.img_paths = img_paths
        self.mask_paths = mask_paths
        self.augment = augment

    def __len__(self):
        return len(self.img_paths)

    def _augment(self, img, msk):
        if random.random() < 0.5:
            img = torch.flip(img, dims=[1])
            if msk is not None:
                msk = torch.flip(msk, dims=[1])
        if random.random() < 0.5:
            img = torch.flip(img, dims=[2])
            if msk is not None:
                msk = torch.flip(msk, dims=[2])
        if random.random() < 0.5:
            img = torch.flip(img, dims=[3])
            if msk is not None:
                msk = torch.flip(msk, dims=[3])

        if random.random() < 0.5:
            k = random.randint(0, 3)
            img = torch.rot90(img, k=k, dims=[2, 3])
            if msk is not None:
                msk = torch.rot90(msk, k=k, dims=[2, 3])

        if random.random() < 0.7:
            scale = 1.0 + random.uniform(-0.1, 0.1)
            shift = random.uniform(-0.1, 0.1)
            img = img * scale + shift

        return img, msk

    def __getitem__(self, idx):
        img, spacing = load_nii(self.img_paths[idx])
        img = zscore_norm(img)

        # (H,W,D) -> (D,H,W)，用于3D卷积
        img = np.transpose(img, (2, 0, 1)).astype(np.float32)
        img = torch.from_numpy(img).unsqueeze(0)

        msk = None
        if self.mask_paths is not None:
            msk_arr, _ = load_nii(self.mask_paths[idx])
            msk_arr = (msk_arr > 0.5).astype(np.float32)
            msk_arr = np.transpose(msk_arr, (2, 0, 1))
            msk = torch.from_numpy(msk_arr).unsqueeze(0)

        if self.augment:
            img, msk = self._augment(img, msk)

        item = {
            'image': img,
            'spacing': torch.tensor(spacing, dtype=torch.float32),
            'id': int(self.img_paths[idx].stem.split('.')[0])
        }

        if msk is not None:
            item['mask'] = msk

        return item


In [5]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class SmallUNet3D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=16):
        super().__init__()
        self.enc1 = ConvBlock(in_ch, base)
        self.pool1 = nn.MaxPool3d(2)
        self.enc2 = ConvBlock(base, base * 2)
        self.pool2 = nn.MaxPool3d(2)

        self.bottleneck = ConvBlock(base * 2, base * 4)

        self.up2 = nn.ConvTranspose3d(base * 4, base * 2, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)
        self.up1 = nn.ConvTranspose3d(base * 2, base, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)

        self.head = nn.Conv3d(base, out_ch, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        b = self.bottleneck(self.pool2(e2))

        d2 = self.up2(b)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.head(d1)


class DiceBCELoss(nn.Module):
    def __init__(self, bce_weight=0.5, smooth=1e-5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.bce_weight = bce_weight
        self.smooth = smooth

    def forward(self, logits, targets):
        bce = self.bce(logits, targets)
        probs = torch.sigmoid(logits)

        probs = probs.reshape(probs.size(0), -1)
        targets = targets.reshape(targets.size(0), -1)

        inter = (probs * targets).sum(dim=1)
        den = probs.sum(dim=1) + targets.sum(dim=1)
        dice = 1.0 - (2.0 * inter + self.smooth) / (den + self.smooth)
        dice = dice.mean()

        return self.bce_weight * bce + (1.0 - self.bce_weight) * dice


In [6]:
train_df = pd.read_csv(TRAIN_CSV)
train_df['patient_id'] = train_df['patient_id'].astype(int)

all_train_imgs = sorted(TRAIN_IMG_DIR.glob('*.nii.gz'))
all_train_msks = [TRAIN_MASK_DIR / p.name for p in all_train_imgs]

print('Train samples:', len(all_train_imgs))

spacings = []
for p in all_train_imgs[:10]:
    _, sp = load_nii(p)
    spacings.append(sp)
print('Example spacings (first 10):')
for s in spacings:
    print(s)


Train samples: 49
Example spacings (first 10):
(np.float32(0.3125), np.float32(0.3125), np.float32(0.4500237))
(np.float32(0.3125), np.float32(0.3125), np.float32(0.40005842))
(np.float32(0.26041666), np.float32(0.26041666), np.float32(0.31999907))
(np.float32(0.28125), np.float32(0.28125), np.float32(0.30000106))
(np.float32(0.3906), np.float32(0.3906), np.float32(0.6001386))
(np.float32(0.26041666), np.float32(0.26041666), np.float32(0.50000197))
(np.float32(0.3125), np.float32(0.31249997), np.float32(0.35008207))
(np.float32(0.26041666), np.float32(0.26041666), np.float32(0.319998))
(np.float32(0.28125), np.float32(0.28125), np.float32(0.30000085))
(np.float32(0.26041666), np.float32(0.26041666), np.float32(0.600001))


In [7]:
CFG = {
    'n_splits': 5,
    'epochs': 25,
    'batch_size': 2,
    'lr': 1e-3,
    'num_workers': 0,
    'threshold': 0.5
}

def train_one_fold(fold, tr_idx, va_idx):
    tr_imgs = [all_train_imgs[i] for i in tr_idx]
    tr_msks = [all_train_msks[i] for i in tr_idx]
    va_imgs = [all_train_imgs[i] for i in va_idx]
    va_msks = [all_train_msks[i] for i in va_idx]

    tr_ds = AneurysmSegDataset(tr_imgs, tr_msks, augment=True)
    va_ds = AneurysmSegDataset(va_imgs, va_msks, augment=False)

    tr_loader = DataLoader(tr_ds, batch_size=CFG['batch_size'], shuffle=True, num_workers=CFG['num_workers'])
    va_loader = DataLoader(va_ds, batch_size=1, shuffle=False, num_workers=CFG['num_workers'])

    model = SmallUNet3D(base=16).to(DEVICE)
    criterion = DiceBCELoss(bce_weight=0.5)
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG['lr'])

    best_vs = -1.0
    best_path = CKPT_DIR / f'fold_{fold}.pt'

    for epoch in range(CFG['epochs']):
        model.train()
        train_losses = []

        for batch in tr_loader:
            imgs = batch['image'].to(DEVICE)
            msks = batch['mask'].to(DEVICE)

            optimizer.zero_grad()
            logits = model(imgs)
            loss = criterion(logits, msks)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_gt, val_pred = [], []

        with torch.no_grad():
            for batch in va_loader:
                img = batch['image'].to(DEVICE)
                msk = batch['mask'].cpu().numpy()[0, 0]
                spacing = batch['spacing'].numpy()[0]

                logits = model(img)
                prob = torch.sigmoid(logits).cpu().numpy()[0, 0]
                pred_mask = (prob > CFG['threshold']).astype(np.uint8)

                gt_vol = volume_from_mask(msk, spacing)
                pred_vol = volume_from_mask(pred_mask, spacing)
                val_gt.append(gt_vol)
                val_pred.append(pred_vol)

        val_vs = volumetric_similarity(val_gt, val_pred)
        print(f'Fold {fold} | Epoch {epoch+1:02d} | TrainLoss {np.mean(train_losses):.4f} | ValVS {val_vs:.4f}')

        if val_vs > best_vs:
            best_vs = val_vs
            torch.save(model.state_dict(), best_path)

    print(f'Fold {fold} best ValVS = {best_vs:.4f}')
    return best_path


In [8]:
# 5折交叉验证训练 + OOF体积评估
kf = KFold(n_splits=CFG['n_splits'], shuffle=True, random_state=SEED)
fold_models = []

for fold, (tr_idx, va_idx) in enumerate(kf.split(all_train_imgs)):
    best_model_path = train_one_fold(fold, tr_idx, va_idx)
    fold_models.append(best_model_path)

print('Saved fold models:')
for p in fold_models:
    print(p)

# 重新做OOF推理，计算整体VS（使用train.csv中的真实体积）
oof_pred = {}

for fold, (_, va_idx) in enumerate(kf.split(all_train_imgs)):
    model = SmallUNet3D(base=16).to(DEVICE)
    model.load_state_dict(torch.load(fold_models[fold], map_location=DEVICE))
    model.eval()

    va_imgs = [all_train_imgs[i] for i in va_idx]
    va_msks = [all_train_msks[i] for i in va_idx]
    va_ds = AneurysmSegDataset(va_imgs, va_msks, augment=False)
    va_loader = DataLoader(va_ds, batch_size=1, shuffle=False)

    with torch.no_grad():
        for batch in va_loader:
            pid = int(batch['id'].item())
            img = batch['image'].to(DEVICE)
            spacing = batch['spacing'].numpy()[0]

            logit = model(img)
            prob = torch.sigmoid(logit).cpu().numpy()[0, 0]
            pred_mask = (prob > CFG['threshold']).astype(np.uint8)
            pred_vol = volume_from_mask(pred_mask, spacing)
            oof_pred[pid] = pred_vol

oof_df = train_df[['patient_id', 'volume']].copy()
oof_df['pred_volume'] = oof_df['patient_id'].map(oof_pred)
oof_vs = volumetric_similarity(oof_df['volume'].values, oof_df['pred_volume'].values)
print('OOF VS =', round(oof_vs, 6))
oof_df.to_csv(PRED_DIR / 'oof_predictions.csv', index=False)


Fold 0 | Epoch 01 | TrainLoss 0.7851 | ValVS 0.2322
Fold 0 | Epoch 02 | TrainLoss 0.7255 | ValVS 0.1761
Fold 0 | Epoch 03 | TrainLoss 0.6945 | ValVS 0.2531
Fold 0 | Epoch 04 | TrainLoss 0.6716 | ValVS 0.1032
Fold 0 | Epoch 05 | TrainLoss 0.6503 | ValVS 0.4222
Fold 0 | Epoch 06 | TrainLoss 0.6302 | ValVS 0.4753
Fold 0 | Epoch 07 | TrainLoss 0.6138 | ValVS 0.6317
Fold 0 | Epoch 08 | TrainLoss 0.5984 | ValVS 0.3314
Fold 0 | Epoch 09 | TrainLoss 0.5842 | ValVS 0.3947
Fold 0 | Epoch 10 | TrainLoss 0.5715 | ValVS 0.2000
Fold 0 | Epoch 11 | TrainLoss 0.5626 | ValVS 0.5937
Fold 0 | Epoch 12 | TrainLoss 0.5477 | ValVS 0.5864
Fold 0 | Epoch 13 | TrainLoss 0.5348 | ValVS 0.3455
Fold 0 | Epoch 14 | TrainLoss 0.5279 | ValVS 0.3041
Fold 0 | Epoch 15 | TrainLoss 0.5220 | ValVS 0.4900
Fold 0 | Epoch 16 | TrainLoss 0.5068 | ValVS 0.0703
Fold 0 | Epoch 17 | TrainLoss 0.5187 | ValVS 0.3725
Fold 0 | Epoch 18 | TrainLoss 0.4940 | ValVS 0.5314
Fold 0 | Epoch 19 | TrainLoss 0.4954 | ValVS 0.4092
Fold 0 | Epo

In [9]:
# 测试集推理：K折模型做概率平均，再阈值化得到mask并计算体积
test_imgs = sorted(TEST_IMG_DIR.glob('*.nii.gz'))
rows_ens = []
rows_per_fold = []

models = []
for fold_id, mp in enumerate(fold_models):
    m = SmallUNet3D(base=16).to(DEVICE)
    m.load_state_dict(torch.load(mp, map_location=DEVICE))
    m.eval()
    models.append((fold_id, m))

for img_path in tqdm(test_imgs, desc='Inference'):
    img_np, spacing = load_nii(img_path)
    img_np = zscore_norm(img_np)
    img_np = np.transpose(img_np, (2, 0, 1)).astype(np.float32)
    img_t = torch.from_numpy(img_np).unsqueeze(0).unsqueeze(0).to(DEVICE)

    pid = int(img_path.stem.split('.')[0])
    fold_probs = []

    with torch.no_grad():
        for fold_id, m in models:
            logit = m(img_t)
            prob = torch.sigmoid(logit).cpu().numpy()[0, 0]
            fold_probs.append(prob)
            fold_mask = (prob > CFG['threshold']).astype(np.uint8)
            fold_volume = volume_from_mask(fold_mask, spacing)
            rows_per_fold.append({'patient_id': pid, 'fold': fold_id, 'volume': float(fold_volume)})

    mean_prob = np.mean(fold_probs, axis=0)
    pred_mask = (mean_prob > CFG['threshold']).astype(np.uint8)
    pred_volume = volume_from_mask(pred_mask, spacing)
    rows_ens.append({'patient_id': pid, 'volume': float(pred_volume)})

sub_ens = pd.DataFrame(rows_ens).sort_values('patient_id').reset_index(drop=True)
sub_per_fold = pd.DataFrame(rows_per_fold).sort_values(['patient_id', 'fold']).reset_index(drop=True)

# 1) 实验目录内保存（用于多实验管理）
sub_exp_path = PRED_DIR / 'submission_ensemble.csv'
sub_ens.to_csv(sub_exp_path, index=False)

wide = sub_per_fold.pivot(index='patient_id', columns='fold', values='volume').reset_index()
wide.columns = ['patient_id'] + [f'volume_fold_{c}' for c in wide.columns[1:]]
blend_path = PRED_DIR / 'blend_features.csv'
wide.to_csv(blend_path, index=False)

# 2) 平台标准输出路径（避免提交报错）
platform_sub = OUTPUT_ROOT / 'submission.csv'
sub_ens.to_csv(platform_sub, index=False)

# 额外兜底：同时写到评测器硬编码路径与/working
for extra in [Path('/root/setup/solution/working/submission.csv'), Path('/working/submission.csv')]:
    try:
        extra.parent.mkdir(parents=True, exist_ok=True)
        sub_ens.to_csv(extra, index=False)
        print('Also saved:', extra)
    except Exception as e:
        print('Skip extra path', extra, 'reason:', e)

print('Experiment submission:', sub_exp_path)
print('Blend features:', blend_path)
print('Platform submission:', platform_sub)
print(sub_ens.head())


Inference:   0%|          | 0/10 [00:00<?, ?it/s]

Experiment submission: /Users/songling/Desktop/Aneurysm Volume Prediction/working/segcv_20260219_182619/predictions/submission_ensemble.csv
Blend features: /Users/songling/Desktop/Aneurysm Volume Prediction/working/segcv_20260219_182619/predictions/blend_features.csv
Platform submission: /Users/songling/Desktop/Aneurysm Volume Prediction/working/submission.csv
   patient_id      volume
0          59  107.361792
1          60  420.898624
2          61   53.108654
3          62  279.899022
4          63  468.343303


## 使用建议
- 本地数据可直接放在 `/Users/songling/Desktop/Aneurysm Volume Prediction`。
- 比赛平台（如图）会自动识别 `/dataset/public` 为输入，`/working/submission.csv` 为标准输出。
- 每次运行会自动创建实验文件夹：`<working>/<RUN_NAME>/`，避免覆盖历史实验。
- 若想自定义实验名，运行前设置：`os.environ['RUN_NAME'] = 'exp_001'`。
- 直接提交用：`/working/submission.csv`（平台读取这个路径）。
- 融合用：`<working>/<RUN_NAME>/predictions/blend_features.csv`。
